<a href="https://colab.research.google.com/github/antariksha-agi/chatbot-Api/blob/main/chatbot-api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
from datetime import datetime
from typing import Optional
from fastapi import FastAPI
from pydantic import BaseModel
from groq import Groq
from google.colab import userdata

Client = Groq(api_key=userdata.get('GROQ_API_KEY'))


# 1. Database Setup
def init_db():
    conn = sqlite3.connect('chatbot.db')
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS interactions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL,
            user_message TEXT NOT NULL,
            bot_reply TEXT NOT NULL,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    conn.commit()
    conn.close()

init_db()

# 2. Pydantic Models
class messages(BaseModel):
    username: str
    user_message: str

app = FastAPI()

@app.post("/send_message/")
def send_message(msg: messages):
  if msg.username and msg.user_message:
    completion = Client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": msg.user_message}
        ]
    )
    bot_reply = completion.choices[0].message.content

    conn = sqlite3.connect('chatbot.db')
    cursor = conn.cursor()
    cursor.execute('''
    INSERT INTO interactions(username, user_message, bot_reply, timestamp)VALUES(?, ?, ?, ?)''', (msg.username, msg.user_message, bot_reply, datetime.now().isoformat()))
    conn.commit()
    conn.close()
    return {"username": msg.username, "user_message": msg.user_message, "bot_reply": bot_reply, "timestamp": datetime.now()}
  else:
    return "Invalid input"

# Create an instance of the messages Pydantic model
message_instance = messages(username="antariksha", user_message="Hello")
mi2 = messages(username="alex", user_message="yoo wussap")
mi3 = messages(username="anjali", user_message="how are you")
input_data = send_message(msg=message_instance)
input_data2 = send_message(msg=mi2)
input_data3 = send_message(msg=mi3)
print(input_data)
print(input_data2)
print(input_data3)

{'username': 'antariksha', 'user_message': 'Hello', 'bot_reply': "Hello. It's nice to meet you. Is there something I can help you with or would you like to chat?", 'timestamp': datetime.datetime(2026, 6, 1, 16, 40, 48, 188126)}
{'username': 'alex', 'user_message': 'yoo wussap', 'bot_reply': "Not much, just here to help. What's on your mind? Need assistance with something or just wanna chat?", 'timestamp': datetime.datetime(2026, 6, 1, 16, 40, 48, 403341)}
{'username': 'anjali', 'user_message': 'how are you', 'bot_reply': "I'm doing well, thanks for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to help with any questions or tasks you may have. How can I assist you today?", 'timestamp': datetime.datetime(2026, 6, 1, 16, 40, 48, 575703)}


In [ ]:
@app.delete("/delete_history/{username}")
def delete_history(username: str):
  conn = sqlite3.connect('chatbot.db')
  cursor = conn.cursor()

  cursor.execute("DELETE FROM interactions WHERE username = ?" , (username,))
  conn.commit()
  conn.close()
  return {"message": "History deleted successfully"}
dh = delete_history("alex")
print(dh)

{'message': 'History deleted successfully'}


In [ ]:
@app.get("/history/")
def get_history():
  conn = sqlite3.connect('chatbot.db')
  cursor = conn.cursor()
  cursor.execute('SELECT * FROM interactions')
  rows = cursor.fetchall()
  conn.close()
  return rows

history = get_history()
print(history)

[(1, 'antariksha', 'Hello', "Hello. It's nice to meet you. Is there something I can help you with or would you like to chat?", '2026-06-01T16:40:48.180245'), (3, 'anjali', 'how are you', "I'm doing well, thanks for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to help with any questions or tasks you may have. How can I assist you today?", '2026-06-01T16:40:48.566962')]


In [ ]:
!pip install groq